# A2: Address Standardization

---

## Overview

Standardize addresses across datasets and link permits by APN and address.

**Inputs:**
- `zoning_permits_*.csv`
- `building_permits_*.csv`

**Outputs:**
- `master_address_registry.csv`
- `permits_linked.csv`

**Key Functions:**
- FIFTH <-> 5TH conversions
- Ave <-> AV <-> Avenue normalization
- Unit/apartment stripping

---

## 1. Setup

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import json

# Add modules to path
sys.path.insert(0, str(Path.cwd().parent.parent))

# Import our modules
from modules.address_normalizer import (
    normalize_address,
    standardize_address,
    parse_address,
    get_street_name_variations,
    get_street_type_variations,
    generate_address_variations
)
from modules.data_loader import load_csv

# Configuration
with open('../../config/berkeley_config.json') as f:
    CONFIG = json.load(f)

DATA_DIR = Path(CONFIG['paths']['data_dir'])

print("Modules loaded successfully")

## 2. Demonstrate Address Normalization

Show how the address normalizer handles common variations.

In [ ]:
# Test addresses
test_addresses = [
    "1914 FIFTH St",
    "1914 5th Street",
    "1914 5TH ST #101",
    "2700 Shattuck Avenue",
    "2700 SHATTUCK Ave",
    "2700 shattuck av",
    "1234 University Ave Apt 5",
]

print("Address Normalization Examples:")
print("="*60)

for addr in test_addresses:
    normalized = normalize_address(addr)
    print(f"Original:   {addr}")
    print(f"Normalized: {normalized}")
    print()

In [ ]:
# Show all variations for an address
print("All variations for '1914 FIFTH ST':")
print("="*60)

variations = generate_address_variations("1914", "FIFTH", "ST")
for v in sorted(variations):
    print(f"  {v}")

print(f"\nTotal variations: {len(variations)}")

## 3. Load Permit Data

In [ ]:
# Find latest permit files
zoning_files = list(DATA_DIR.glob('zoning_permits_*.csv'))
building_files = list(DATA_DIR.glob('building_permits_*.csv'))

print("Available permit files:")
for f in zoning_files + building_files:
    print(f"  {f.name}")

# Load most recent
df_zoning = None
df_building = None

if zoning_files:
    latest_zoning = max(zoning_files, key=lambda x: x.stat().st_mtime)
    df_zoning = load_csv(latest_zoning)

if building_files:
    latest_building = max(building_files, key=lambda x: x.stat().st_mtime)
    df_building = load_csv(latest_building)

# Also try to load existing housing projects
housing_path = DATA_DIR / 'housing_projects_FINAL.csv'
if housing_path.exists():
    df_housing = load_csv(housing_path)
    print(f"\nLoaded housing projects: {len(df_housing)} records")

## 4. Standardize Addresses in Datasets

In [ ]:
def add_normalized_address(df, address_col='address'):
    """
    Add normalized address column to dataframe.
    """
    if address_col not in df.columns:
        # Try common variations
        for col in ['address_display', 'b1_full_address', 'site_address']:
            if col in df.columns:
                address_col = col
                break
    
    if address_col in df.columns:
        df = df.copy()
        df['address_normalized'] = df[address_col].apply(normalize_address)
        print(f"Added normalized addresses from '{address_col}'")
        return df, address_col
    else:
        print(f"Warning: No address column found")
        print(f"Available columns: {df.columns.tolist()}")
        return df, None

# Add normalized addresses to housing data if loaded
if 'df_housing' in dir() and df_housing is not None:
    df_housing, addr_col = add_normalized_address(df_housing, 'address_display')
    
    print("\nSample normalized addresses:")
    sample = df_housing[['address_display', 'address_normalized']].head(10)
    display(sample)

## 5. Create Master Address Registry

Combine all unique addresses from all datasets.

In [ ]:
# Collect all unique addresses
all_addresses = set()
address_sources = {}

# From housing projects
if 'df_housing' in dir() and df_housing is not None:
    for addr in df_housing['address_normalized'].dropna().unique():
        all_addresses.add(addr)
        address_sources[addr] = address_sources.get(addr, []) + ['housing']

print(f"Unique addresses collected: {len(all_addresses)}")

# Create registry dataframe
registry_data = []
for addr in all_addresses:
    parsed = parse_address(addr)
    registry_data.append({
        'address_normalized': addr,
        'street_number': parsed['street_number'],
        'street_name': parsed['street_name'],
        'street_type': parsed['street_type'],
        'sources': ','.join(address_sources.get(addr, []))
    })

df_registry = pd.DataFrame(registry_data)
print(f"\nMaster registry: {len(df_registry)} addresses")
display(df_registry.head(10))

## 6. Link Permits by APN and Address

Connect related permits across datasets.

In [ ]:
def normalize_apn(apn):
    """
    Normalize APN for matching.
    """
    if pd.isna(apn):
        return None
    # Remove spaces and standardize format
    apn_str = str(apn).strip()
    # Remove any non-alphanumeric except spaces
    apn_clean = ''.join(c for c in apn_str if c.isalnum() or c == ' ')
    return apn_clean

# Add normalized APN to housing data
if 'df_housing' in dir() and df_housing is not None and 'apn' in df_housing.columns:
    df_housing['apn_normalized'] = df_housing['apn'].apply(normalize_apn)
    
    # Count unique APNs
    unique_apns = df_housing['apn_normalized'].dropna().nunique()
    print(f"Unique APNs: {unique_apns}")
    
    # Show sample
    print("\nSample APNs:")
    display(df_housing[['address_display', 'apn', 'apn_normalized']].head(10))

## 7. Export Results

In [ ]:
# Export master address registry
registry_path = DATA_DIR / 'master_address_registry.csv'
df_registry.to_csv(registry_path, index=False)
print(f"Saved: {registry_path}")

# Export housing projects with normalized addresses
if 'df_housing' in dir() and df_housing is not None:
    housing_normalized_path = DATA_DIR / 'housing_projects_normalized.csv'
    df_housing.to_csv(housing_normalized_path, index=False)
    print(f"Saved: {housing_normalized_path}")

print("\nAddress standardization complete!")

---

## Summary

This notebook:
- Demonstrated address normalization (FIFTH<->5TH, Ave<->AV)
- Standardized addresses across datasets
- Created master address registry
- Normalized APNs for linking

**Next:** Run `A3_geocoding_pipeline.ipynb` to add coordinates.